In [ ]:
import sys
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from cell2location.models import RegressionModel, Cell2location

sys.path.append(os.path.abspath("src"))
from loaders import (
    DataCenter, DataSource, DataFile, SourceNames,
    SingleCellLoader, VisiumLoader,
)

DATA_PATH = "/path/to/data_folder"
RESULTS_FOLDER = "/path/to/results"

mosaic_data = DataCenter(
    name="MOSAIC",
    sources={
        SourceNames.sc_rna: DataSource(
            files={"scRNA anndata": DataFile(name=f"{DATA_PATH}/single_cell/reference_atlas.h5ad", loader=SingleCellLoader())}
        ),
        SourceNames.spatial: DataSource(
            files={"Visium anndata": DataFile(name=f"{DATA_PATH}/spatial/visium_data", loader=VisiumLoader())}
        )
    }
)

In [ ]:
def run_spot_deconvolution(sample_id, cell_types, filter_method, plot_QC_and_fitting, accelerator):
    """
    Runs the spot deconvolution pipeline for a given sample using celltype_level4_scanvi.

    This function performs the following steps:
      1. Creates a folder to save per-sample results.
      2. Loads and processes the single-cell data for the specified sample.
      3. Filters genes based on the provided filtering method.
      4. Sets up and trains a reference regression model to derive cell type signatures.
      5. Saves the trained reference model and exports posterior estimates.
      6. Extracts cell type signature matrix (inf_aver) from the processed single-cell data.
      7. Loads the corresponding spatial (Visium) data.
      8. Identifies and subsets to the common genes between single-cell and spatial datasets.
      9. Saves the cell type signature matrix.
      10. Sets up and trains the Cell2location spatial deconvolution model.
      11. Saves the trained spatial model and exports its posterior estimates.
      12. Writes the deconvolved spatial AnnData object to disk.

    Args:
        sample_id (str): The sample identifier (e.g., 'HK_G_080a'). Used to locate and store sample-specific results.
        filter_method (str): The gene filtering method to use. Must be either 'HVG' or 'filter_genes'.
        plot_QC_and_fitting (bool): If True, generates quality control and fitting diagnostic plots.
        accelerator (str): Accelerator for model training (e.g., 'cpu' or 'gpu').

    Raises:
        ValueError: If sample_id is not a non-empty string.
        ValueError: If filter_method is not 'HVG' or 'filter_genes'.
        ValueError: If plot_QC_and_fitting is not a boolean.
        ValueError: If accelerator is not a non-empty string.
    """

    # Ensure required arguments are defined properly
    if not sample_id or not isinstance(sample_id, str):
        raise ValueError("sample_id must be a non-empty string")
    if filter_method not in ['HVG', 'filter_genes']:
        raise ValueError("filter_method must be either 'HVG' or 'filter_genes'")
    if not isinstance(plot_QC_and_fitting, bool):
        raise ValueError("plot_QC_and_fitting must be a boolean")
    if not accelerator or not isinstance(accelerator, str):
        raise ValueError("accelerator must be a non-empty string")

    # Folder to save results for the sample
    base_folder = '/home/sagemaker-user/user-default-efs/spot_deconvolution/GBMap_cell_types/per_sample/'
    folder_path = os.path.join(base_folder, sample_id)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created folder: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")

    # Convert sample_id to a list for subsetting single-cell data
    sample_id_list = [sample_id]

    # Access single cell data (assumes 'single_cell_obj' is defined in the scope)
    adata_sc = adata[adata.obs['sample_id'].isin(sample_id_list)].copy()

    # Filter genes based on specified method
    if filter_method == 'HVG':
        # Select highly variable genes using Seurat flavor based on the 'LogNormalize' layer
        sc.pp.highly_variable_genes(adata_sc, flavor='seurat', n_top_genes=2000, layer='LogNormalize')
        # Subset to highly variable genes
        adata_sc = adata_sc[:, adata_sc.var.highly_variable].copy()
    elif filter_method == 'filter_genes':
        # Use a custom filtering function (assumed to be defined elsewhere)
        selected = filter_genes(adata_sc, cell_count_cutoff=5, cell_percentage_cutoff2=0.05, nonz_mean_cutoff=1.0)
        adata_sc = adata_sc[:, selected].copy()

    print(f"Filtered single-cell data shape: {adata_sc.shape}")  # Should be smaller after filtering

    # Setup AnnData for the reference (regression) model using the specified cell type annotations
    RegressionModel.setup_anndata(adata_sc, labels_key=cell_types)  # Adjust key if needed

    # Train the reference regression model
    mod = RegressionModel(adata_sc)
    mod.train(max_epochs=250, accelerator=accelerator)

    # Save the trained regression model
    mod_path = os.path.join(folder_path, 'regression_model/')
    mod.save(mod_path)

    # Export the posterior estimates (reference cell type signatures)
    adata_sc = mod.export_posterior(adata_sc, sample_kwargs={'num_samples': 1000})

    # Optionally, generate QC and fitting diagnostic plots for the reference model
    if plot_QC_and_fitting:
        mod.plot_history(20)
        mod.plot_QC()

    # Extract reference cell type signatures as a pandas DataFrame (cell type signature matrix)
    if 'means_per_cluster_mu_fg' in adata_sc.varm.keys():
        inf_aver = adata_sc.varm['means_per_cluster_mu_fg'][
            [f'means_per_cluster_mu_fg_{i}' for i in adata_sc.uns['mod']['factor_names']]
        ].copy()
    else:
        inf_aver = adata_sc.var[
            [f'means_per_cluster_mu_fg_{i}' for i in adata_sc.uns['mod']['factor_names']]
        ].copy()
    inf_aver.columns = adata_sc.uns['mod']['factor_names']

    def to_visium_id(sample_id):
        """
        Converts sample ID from format 'HKG001a' → 'HK_G_001a_vis'.
        Works for any 3-digit number and optional trailing letter.
        """
        match = re.match(r"HKG(\d{3}[a-zA-Z])", sample_id)
        if match:
            return f"HK_G_{match.group(1)}_vis"
        else:
            raise ValueError(f"Unexpected sample_id format: {sample_id}")

    visium_sample_id = to_visium_id(sample_id)

    # Load Visium (spatial) data for the sample
    visium_obj = MosaicDataset.load_visium(
        sample_list=[visium_sample_id],
        resolution="hires"
    )
    adata_vis = visium_obj[visium_sample_id]

    # Identify common genes between the spatial and cell type signature datasets
    common_genes = np.intersect1d(adata_vis.var_names, inf_aver.index)
    print(f"Number of common genes: {len(common_genes)}")

    # Subset the Visium data and the signature matrix to the common genes
    adata_vis = adata_vis[:, common_genes].copy()
    inf_aver = inf_aver.loc[common_genes, :].copy()

    # save the cell type signature matrix to a compressed CSV file
    inf_aver.to_csv(os.path.join(folder_path, 'cell_type_signatures.csv.gz'), index=False, compression='gzip')

    print(f"Visium shape: {adata_vis.shape}")  # Should have only the common genes

    # Setup AnnData for the Cell2location spatial deconvolution model
    Cell2location.setup_anndata(adata_vis)

    # Initialize and train the Cell2location model
    mod_sp = Cell2location(adata_vis, cell_state_df=inf_aver, N_cells_per_location=5)
    mod_sp.train(max_epochs=300, accelerator=accelerator)  # Change accelerator if needed
 
    # Save the trained Cell2location model
    mod_sp_path = os.path.join(folder_path, 'cell2location_model/')
    mod_sp.save(mod_sp_path)

    # Export the posterior estimates (cell type abundances) from the spatial model
    adata_vis = mod_sp.export_posterior(adata_vis, sample_kwargs={'num_samples': 1000})

    # Optionally, generate QC and fitting diagnostic plots for the spatial model
    if plot_QC_and_fitting:
        mod_sp.plot_history(20)
        mod_sp.plot_QC()

    # Add the posterior cell type abundance estimates to adata_vis.obs
    adata_vis.obs[adata_vis.uns['mod']['factor_names']] = adata_vis.obsm['q05_cell_abundance_w_sf']

    # Save the deconvolved spatial AnnData object to disk
    adata_vis.write(os.path.join(folder_path, 'adata_vis_deconvolved.h5ad'))


In [ ]:
# Get list of all available samples from directory
all_samples = os.listdir(f"{DATA_PATH}/spatial/visium_data")

for sample in all_samples:
    run_spot_deconvolution(
        sample_id=sample,
        cell_types='cluster_id',
        filter_method='filter_genes',
        plot_QC_and_fitting=False,
        accelerator='gpu'  # Change to 'cpu' if no GPU available
    )

In [ ]:
adata = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap/sc_mosaic_rounded_GBmap.h5ad")

In [ ]:
for sample_id in adata.obs['sample_id'].cat.categories[1:]:
    run_spot_deconvolution(sample_id=sample_id,
                            cell_types='cluster_id',
                            filter_method='filter_genes',
                            plot_QC_and_fitting=False,
                            accelerator='gpu')